<a href="https://colab.research.google.com/github/firdosezahra/CNN_MRI/blob/main/cnn_mri.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =====================================================
#  STEP 1:  Install / Import Libraries
# =====================================================
!pip install tensorflow  # already available on Colab, but safe to run

import tensorflow as tf
import numpy as np
import os
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

print("✅ TensorFlow version:", tf.__version__)

✅ TensorFlow version: 2.19.0


In [ ]:
# =====================================================
#  STEP 2:  Mount Google Drive & Define Paths
# =====================================================
from google.colab import drive
drive.mount('/content/drive')

# Change this path to where you uploaded your dataset on Drive
BASE_DIR = "/content/drive/MyDrive/brain cancer/data split"
TRAIN_DIR = os.path.join(BASE_DIR, "train")
VAL_DIR   = os.path.join(BASE_DIR, "val")
TEST_DIR  = os.path.join(BASE_DIR, "test")

print("Dataset folders ready.")


ValueError: mount failed

In [ ]:
# =====================================================
#  STEP 3:  Data Preprocessing & Augmentation
# =====================================================
train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

val_test_gen = ImageDataGenerator(rescale=1./255)

train_data = train_gen.flow_from_directory(
    TRAIN_DIR, target_size=(224,224), batch_size=32, class_mode='categorical'
)
val_data = val_test_gen.flow_from_directory(
    VAL_DIR, target_size=(224,224), batch_size=32, class_mode='categorical'
)
test_data = val_test_gen.flow_from_directory(
    TEST_DIR, target_size=(224,224), batch_size=32, class_mode='categorical'
)

print(" Data generators ready.")


In [ ]:
# =====================================================
#  STEP 4:  Build VGG16 Model (Transfer Learning)
# =====================================================
import tensorflow as tf
import numpy as np
import os
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from google.colab import drive
from tensorflow.keras.utils import load_img, img_to_array

# Change this path to where you uploaded your dataset on Drive
BASE_DIR = "/content/drive/MyDrive/brain cancer/data split"
TRAIN_DIR = os.path.join(BASE_DIR, "train")
VAL_DIR   = os.path.join(BASE_DIR, "val")
TEST_DIR  = os.path.join(BASE_DIR, "test")

train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

val_test_gen = ImageDataGenerator(rescale=1./255)

train_data = train_gen.flow_from_directory(
    TRAIN_DIR, target_size=(224,224), batch_size=32, class_mode='categorical'
)
val_data = val_test_gen.flow_from_directory(
    VAL_DIR, target_size=(224,224), batch_size=32, class_mode='categorical'
)
test_data = val_test_gen.flow_from_directory(
    TEST_DIR, target_size=(224,224), batch_size=32, class_mode='categorical'
)


base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224,224,3))
for layer in base_model.layers:
    layer.trainable = False  # freeze pretrained weights

x = GlobalAveragePooling2D()(base_model.output)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
output = Dense(len(train_data.class_indices), activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=output)

model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# checkpoint = ModelCheckpoint("best_model.keras", monitor='val_accuracy', save_best_only=True, mode='max')
# earlystop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# history = model.fit(
#     train_data,
#     validation_data=val_data,
#     epochs=15,
#     callbacks=[checkpoint, earlystop]
# )

test_loss, test_acc = model.evaluate(test_data)
print(f"✅ Test Accuracy: {test_acc*100:.2f}%")



# Corrected model saving path
model_save_path = "/content/drive/MyDrive/brain cancer/brain_tumor_model_final.keras"
model.save(model_save_path)

print(f"💾 Model saved successfully to {model_save_path}.")

img_path = "/content/drive/MyDrive/brain cancer/data split/test/glioma_tumor/gg (37).jpg"
# img_path = "/content/drive/MyDrive/brain cancer/data split/test/pituitary_tumor/p (91).jpg"
# img_path = "/content/drive/MyDrive/brain cancer/data split/test/pituitary_tumor/p (129).jpg"
# img_path = "/content/drive/MyDrive/brain cancer/data split/test/pituitary_tumor/p (93).jpg"
# img_path = "/content/drive/MyDrive/brain cancer/data split/test/pituitary_tumor/p (77).jpg"
# img_path = "/content/drive/MyDrive/brain cancer/data split/test/pituitary_tumor/p (66).jpg"
# img_path = "/content/drive/MyDrive/brain cancer/data split/test/meningioma_tumor/m (15).jpg"
# img_path = "/content/drive/MyDrive/brain cancer/data split/test/meningioma_tumor/m (29).jpg"
# img_path = "/content/drive/MyDrive/brain cancer/data split/test/meningioma_tumor/m (84).jpg"
# img_path = "/content/drive/MyDrive/brain cancer/data split/test/meningioma_tumor/m (97).jpg"


img = load_img(img_path, target_size=(224,224))
x = img_to_array(img)/255.0
x = np.expand_dims(x, axis=0)

pred = model.predict(x)
classes = list(train_data.class_indices.keys())
predicted_class = classes[np.argmax(pred)]
confidence = np.max(pred)*100

print(f"🧩 Predicted Class: {predicted_class}")
print(f"Confidence: {confidence:.2f}%")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from tensorflow.keras.models import load_model
model = load_model('/content/drive/MyDrive/brain_cancer/brain_tumor_model_final.keras')


test_loss, test_acc = model.evaluate(test_data)
print(f" Test Accuracy: {test_acc*100:.2f}%")


# Corrected model saving path
model_save_path = "/content/drive/MyDrive/brain_cancer/brain_tumor_model_final.keras"
model.save(model_save_path)

print(f" Model saved successfully to {model_save_path}.")

# img_path = "/content/drive/MyDrive/brain cancer/data split/test/glioma_tumor/gg (37).jpg"
img_path = "/content/drive/MyDrive/brain cancer/data split/test/pituitary_tumor/p (91).jpg"
# img_path = "/content/drive/MyDrive/brain cancer/data split/test/pituitary_tumor/p (129).jpg"
# img_path = "/content/drive/MyDrive/brain cancer/data split/test/pituitary_tumor/p (93).jpg"
# img_path = "/content/drive/MyDrive/brain cancer/data split/test/pituitary_tumor/p (77).jpg"
# img_path = "/content/drive/MyDrive/brain cancer/data split/test/pituitary_tumor/p (66).jpg"
# img_path = "/content/drive/MyDrive/brain cancer/data split/test/meningioma_tumor/m (15).jpg"
# img_path = "/content/drive/MyDrive/brain cancer/data split/test/meningioma_tumor/m (29).jpg"
# img_path = "/content/drive/MyDrive/brain cancer/data split/test/meningioma_tumor/m (84).jpg"
# img_path = "/content/drive/MyDrive/brain cancer/data split/test/meningioma_tumor/m (97).jpg"


img = load_img(img_path, target_size=(224,224))
x = img_to_array(img)/255.0
x = np.expand_dims(x, axis=0)

pred = model.predict(x)
classes = list(train_data.class_indices.keys())
predicted_class = classes[np.argmax(pred)]
confidence = np.max(pred)*100

print(f" Predicted Class: {predicted_class}")
print(f"Confidence: {confidence:.2f}%")

In [ ]:
# =====================================================
#  STEP 9:  Grad-CAM Visualization (Tumor Heatmap)
# =====================================================
import matplotlib.pyplot as plt
import cv2
from google.colab import drive
drive.mount('/content/drive')


def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    grad_model = tf.keras.models.Model(
        [model.inputs], [model.get_layer(last_conv_layer_name).output, model.output]
    )
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy()

# -----------------------------------
# 1️⃣ Load the same image again
# -----------------------------------
img_path = "/content/drive/MyDrive/brain cancer/data split/test/pituitary_tumor/p (91).jpg"
img = tf.keras.utils.load_img(img_path, target_size=(224,224))
img_array = tf.keras.utils.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

# -----------------------------------
# 2️⃣ Generate heatmap
# -----------------------------------
last_conv_layer_name = "block5_conv3"   # last convolutional layer in VGG16
heatmap = make_gradcam_heatmap(img_array, model, last_conv_layer_name)

# -----------------------------------
# 3️⃣ Overlay heatmap on original image
# -----------------------------------
img_cv = cv2.imread(img_path)
img_cv = cv2.resize(img_cv, (224, 224))
heatmap = cv2.resize(heatmap, (img_cv.shape[1], img_cv.shape[0]))
heatmap = np.uint8(255 * heatmap)
heatmap_color = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
superimposed_img = cv2.addWeighted(img_cv, 0.6, heatmap_color, 0.4, 0)

# -----------------------------------
# 4️⃣ Display Results
# -----------------------------------
plt.figure(figsize=(10,4))
plt.subplot(1,3,1)
plt.title("Original MRI Image")
plt.imshow(cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB))
plt.axis("off")

plt.subplot(1,3,2)
plt.title("Grad-CAM Heatmap")
plt.imshow(heatmap, cmap='jet')
plt.axis("off")

plt.subplot(1,3,3)
plt.title("Overlayed Image")
plt.imshow(cv2.cvtColor(superimposed_img, cv2.COLOR_BGR2RGB))
plt.axis("off")

plt.show()


Mounted at /content/drive


NameError: name 'model' is not defined